## PPO from Stable-Baselines3

`probatum.sb3.wrap` hands the environment to Stable-Baselines3 as one of its own vector
environments, the agent spec riding on its action space, so a stock PPO trains on it and the
trained model is backtested as it trained. Trained on Bitcoin's hourly perpetual at Bybit
from January 2024 to June 2025, then run on the six months after, beside holding the
perpetual long and an agent that acts at random.

In [1]:
import json
import pathlib

import exchange_router
import numpy
import probatum
import probatum.sb3
import stable_baselines3
import stable_baselines3.common.vec_env
import torch

print(f"probatum {probatum.__version__}, exchange-router {exchange_router.__version__}, "
      f"stable-baselines3 {stable_baselines3.__version__}, torch {torch.__version__}")

probatum 4.0.0, exchange-router 4.0.0, stable-baselines3 2.9.2, torch 2.15.0+cpu


### The bars

In [2]:
PATH = pathlib.Path("../bars/bybit-linear-btcusdt-ethusdt-1h-2024-2025")

if PATH.exists():
    perp = probatum.Data.load(path=PATH)
else:
    with exchange_router.Router.local(exchanges=["bybit"]) as router:
        perp = probatum.Data.from_router(router, exchange="bybit", market="linear",
                                         interval="1h", symbols=["BTCUSDT", "ETHUSDT"],
                                         start="2024-01-01", end="2025-12-31", funding=True)
    perp.save(path=PATH)

train, test = perp["BTCUSDT"].split(when="2025-07-01")

print(train)
print(test)

Data(BTCUSDT, 1h, 13128 bars from 2024-01-01 00:00 to 2025-06-30 23:00)
Data(BTCUSDT, 1h, 4416 bars from 2025-07-01 00:00 to 2025-12-31 23:00)


### The environment and SB3's wrappers

Eight accounts in episodes of two weeks, seen through a window of 32 bars. SB3's own
wrappers compose on top unchanged: one to log the episodes and one to normalise the
observation and the reward. `SpecCallback` copies the normaliser's statistics of the
observation into the model's spec, so a backtest scales the observation as training did.

In [3]:
env = probatum.env(train, venue="bybit_perp", window=32, num_envs=8,
                   episode_length=24 * 14, seed=0)

venv = stable_baselines3.common.vec_env.VecNormalize(
    stable_baselines3.common.vec_env.VecMonitor(probatum.sb3.wrap(env)))

### Training

A stock PPO with its default network, seeded, 300,000 steps on the CPU.

In [4]:
torch.manual_seed(0)

agent = stable_baselines3.PPO(policy="MlpPolicy", env=venv, n_steps=1_024, batch_size=256,
                              seed=0, device="cpu", verbose=0)
agent.learn(total_timesteps=300_000, callback=probatum.sb3.SpecCallback())

episodes = [episode["r"] for episode in agent.ep_info_buffer]
print(f"the last {len(episodes)} training episodes, mean reward {numpy.mean(episodes):.4f}")

the last 100 training episodes, mean reward 0.0921


### On bars it never saw

The model carries its spec, so `backtest` runs it as one environment of that spec over the
test bars from their first close, asking it for its deterministic action at each.

In [5]:
result = probatum.backtest(agent, data=test, venue="bybit_perp")

result

,run,buy_and_hold,low,high,level,method,why_none
total_return_pct,-51.85,-21.45,-67.57,-28.18,0.95,"studentized stationary block bootstrap, blocks of 11 bars",
cagr_pct,,,,,,,1
volatility_pct,26.26,40.07,24.37,28.81,0.95,"studentized stationary block bootstrap, blocks of 11 bars",
sharpe,-5.39,-0.99,-8.28,-2.50,0.95,"delta method, Newey-West lag 11",
max_drawdown_pct,52.06,35.89,34.36,68.70,0.95,"middle of resampled histories, stationary block bootstrap, blocks of 11 bars",
exposure_pct,99.14,,,,,,
num_trades,"1,104",,,,,,
win_rate,0.41,,0.38,0.44,0.95,percentile bootstrap of round trips,
profit_factor,0.62,,0.50,0.77,0.95,BCa bootstrap of round trips,
costs,"7,426",,,,,,


### Beside holding and acting at random

Holding the perpetual long the whole time, and a function of the observation that returns a
weight drawn at random, run on the same spec.

In [6]:
draws = numpy.random.default_rng(seed=0)


def at_random(observation):
    return draws.uniform(low=-1.0, high=1.0, size=(1, 1)).astype(numpy.float32)


held   = probatum.backtest(numpy.ones(shape=len(test)), data=test, venue="bybit_perp")
chance = probatum.backtest(at_random, data=test, venue="bybit_perp", spec=env)

statistics = ["total_return_pct", "sharpe", "max_drawdown_pct", "exposure_pct",
              "num_trades", "funding_paid"]

probatum.compare({"PPO": result, "long": held, "at random": chance},
                 keys=statistics).to_pandas().set_index(keys="name")[statistics].round(
    decimals=2)

,total_return_pct,sharpe,max_drawdown_pct,exposure_pct,num_trades,funding_paid
name,,,,,,
PPO,-51.85,-5.39,52.06,99.14,1104,13.62
long,-20.95,-0.99,35.31,100.00,0,301.11
at random,-80.07,-13.77,80.35,98.94,2266,3.95


### What it decided

`result.policy` holds the weight the agent asked for at each close, which played again as
an array gives the same equity.

In [7]:
result.chart(probatum.chart.marks.Line(result.policy[:, 0], name="weight asked",
                                       panel="weight", step=True),
             focus=24 * 30,
             title="PPO on BTCUSDT perpetual 1h, July to December 2025")

Chart(bars=4416, panels=4)

### Saved beside its spec

`probatum.save` writes the model as its own zip beside the spec it trained with, which
holds the window, the observation, the actions, the reward, the venue, the training span
and the normaliser's statistics.

In [8]:
probatum.save(agent, path="../saved/ppo")

spec = json.loads(pathlib.Path("../saved/ppo/spec.json").read_text(encoding="utf-8"))

print(sorted(path.name for path in pathlib.Path("../saved/ppo").iterdir()))

{key: spec[key] for key in ["window", "instruments", "actions", "reward", "venue",
                            "training"]}

['agent.zip', 'spec.json']


{'window': 32,
 'instruments': ['BTCUSDT'],
 'actions': {'space': 'box', 'low': [-1.0], 'high': 1.0, 'max_leverage': 1.0},
 'reward': {'builtin': 'log', 'floor': 0.01},
 'venue': {'name': 'bybit_perp', 'taken': '2026-11-08', 'changed': {}},
 'training': {'fingerprint': 'b2:7adbda2981977b99c2c5c64d18c853f0',
  'interval': '1h',
  'bars': 13128,
  'start': '2024-01-01T00:00:00Z',
  'end': '2025-06-30T23:00:00Z'}}